<a href="https://colab.research.google.com/github/Oruntu-Tanima-Proje/otProje/blob/main/notebooks/04_resnet50_egitim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 04 - ResNet50 Eğitimi

## Klasik Derin Mimari, Yüksek Doğruluk

Bu notebook, **ResNet50** mimarisini transfer learning ile domates yaprağı
hastalık sınıflandırma problemine uyarlar.

### Model Özellikleri
- **Mimari**: ResNet50 (Microsoft Research, 2015)
- **Parametre**: ~24 milyon
- **Boyut**: ~204 MB
- **Avantaj**: Derin yapı, yüksek doğruluk

### ⚠️ Önemli: ResNet50 Özel Preprocessing
ResNet50, ImageNet'te **caffe-style preprocessing** ile eğitilmiştir.
Bu yüzden `rescale=1./255` yerine `tensorflow.keras.applications.resnet50.preprocess_input`
fonksiyonunu kullanmamız gerekiyor.

Yanlış preprocessing → Model %20 accuracy verir ❌  
Doğru preprocessing → Model %98+ accuracy verir ✅

### Eğitim Stratejisi
1. **Phase 1**: Feature Extraction (10 epoch)
2. **Phase 2**: Fine-Tuning (10 epoch)

### Beklenen Sonuç
- Test Accuracy: ~%98
- Eğitim Süresi: ~85 dakika (T4 GPU)

In [ ]:
# ============================================================
# 1. HAZIRLIK
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 10
drive_proje = "/content/drive/MyDrive/Domates_Projesi"

# Veriyi Drive'dan kopyala (yoksa)
if not os.path.exists("tomato_data"):
    print("📦 Veri seti Drive'dan kopyalanıyor...")
    shutil.copytree(f"{drive_proje}/data", "tomato_data")
    print("   ✅ Tamamlandı")
else:
    print("✅ Veri seti yerinde")

print(f"\n🖥️  GPU sayısı: {len(tf.config.list_physical_devices('GPU'))}")

In [ ]:
# ============================================================
# 2. RESNET50 İÇİN ÖZEL DATAGENERATOR
# preprocess_input fonksiyonu ImageNet formatına çevirir
# ============================================================

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

# ResNet50'ye özel preprocessing ile generator'lar
train_datagen = ImageDataGenerator(
    preprocessing_function=resnet_preprocess,  # ← KRİTİK FARK!
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    fill_mode='nearest'
)

valid_datagen = ImageDataGenerator(preprocessing_function=resnet_preprocess)
test_datagen = ImageDataGenerator(preprocessing_function=resnet_preprocess)

train_generator = train_datagen.flow_from_directory(
    "tomato_data/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

valid_generator = valid_datagen.flow_from_directory(
    "tomato_data/valid",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    "tomato_data/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("\n✅ ResNet50 generator'ları hazır (doğru preprocessing ile)")
print(f"   Train: {train_generator.samples} görüntü")
print(f"   Valid: {valid_generator.samples} görüntü")
print(f"   Test:  {test_generator.samples} görüntü")

In [ ]:
# ============================================================
# 3. RESNET50 MODEL KURULUMU
# ============================================================

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

print("🔨 ResNet50 modeli kuruluyor...")

# 1. ImageNet ağırlıklarıyla ResNet50'yi yükle
resnet_base = ResNet50(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# 2. Base modeli dondur
resnet_base.trainable = False

# 3. Üstüne sınıflandırma katmanları ekle
x = resnet_base.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

# 4. Modeli oluştur
resnet_model = Model(inputs=resnet_base.input, outputs=predictions)

# 5. Compile
resnet_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"\n✅ ResNet50 hazır")
print(f"   Toplam parametre: {resnet_model.count_params():,}")
print(f"   Toplam katman: {len(resnet_base.layers)}")

trainable = sum([tf.size(w).numpy() for w in resnet_model.trainable_weights])
print(f"   Eğitilebilir: {trainable:,}")
print(f"   Donmuş: {resnet_model.count_params() - trainable:,}")

In [ ]:
# ============================================================
# 4. PHASE 1: FEATURE EXTRACTION (10 epoch)
# ============================================================

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import time

os.makedirs("models", exist_ok=True)

callbacks_phase1 = [
    ModelCheckpoint(
        'models/resnet50_phase1.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1)
]

print("=" * 70)
print("🚀 PHASE 1: Feature Extraction")
print("=" * 70)
print("Süreç: ResNet50 base donmuş, üst katmanlar eğitiliyor")
print("Beklenen süre: ~45 dakika\n")

start = time.time()

resnet_history_p1 = resnet_model.fit(
    train_generator,
    epochs=10,
    validation_data=valid_generator,
    callbacks=callbacks_phase1,
    verbose=1
)

elapsed = time.time() - start
print(f"\n✅ Phase 1 tamamlandı! Süre: {elapsed/60:.1f} dakika")
print(f"   En iyi val_accuracy: {max(resnet_history_p1.history['val_accuracy']):.4f}")

In [ ]:
# ============================================================
# 5. PHASE 2 HAZIRLIK: FINE-TUNING
# ============================================================

# Son 30 katmanı eğitilebilir yap
resnet_base.trainable = True

total_layers = len(resnet_base.layers)
print(f"ResNet50 toplam katman: {total_layers}")

fine_tune_at = total_layers - 30

for layer in resnet_base.layers[:fine_tune_at]:
    layer.trainable = False

# Düşük learning rate ile yeniden compile
resnet_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

trainable_params = sum([tf.size(w).numpy() for w in resnet_model.trainable_weights])
print(f"\n✅ Fine-tuning için hazır")
print(f"   Eğitilebilir parametre: {trainable_params:,}")
print(f"   Donmuş katman: {fine_tune_at}")
print(f"   Açık katman: 30")

In [ ]:
# ============================================================
# 6. PHASE 2: FINE-TUNING (10 epoch)
# ============================================================

callbacks_phase2 = [
    ModelCheckpoint(
        'models/resnet50_final.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-8, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1)
]

print("=" * 70)
print("🚀 PHASE 2: Fine-Tuning")
print("=" * 70)
print("Süreç: Son 30 katman + üst katmanlar açık, düşük LR")
print("Beklenen süre: ~45 dakika\n")

start = time.time()

resnet_history_p2 = resnet_model.fit(
    train_generator,
    epochs=10,
    validation_data=valid_generator,
    callbacks=callbacks_phase2,
    verbose=1
)

elapsed = time.time() - start
print(f"\n✅ Phase 2 tamamlandı! Süre: {elapsed/60:.1f} dakika")
print(f"   En iyi val_accuracy: {max(resnet_history_p2.history['val_accuracy']):.4f}")

print(f"\n📊 İyileşme:")
phase1_best = max(resnet_history_p1.history['val_accuracy'])
phase2_best = max(resnet_history_p2.history['val_accuracy'])
print(f"   Phase 1: {phase1_best*100:.2f}%")
print(f"   Phase 2: {phase2_best*100:.2f}%")

In [ ]:
# ============================================================
# 7. TEST SETİ DEĞERLENDİRMESİ
# ============================================================

from tensorflow.keras.models import load_model
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
import numpy as np

print("ResNet50 final modeli yükleniyor...")
resnet_best = load_model('models/resnet50_final.keras')

# Test setinde değerlendir
print("\nTest setinde değerlendiriliyor...")
test_generator.reset()
test_loss, test_accuracy = resnet_best.evaluate(test_generator, verbose=1)

# Tahminler
test_generator.reset()
predictions = resnet_best.predict(test_generator, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

# Metrikler
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')
model_size = os.path.getsize('models/resnet50_final.keras') / (1024 * 1024)

# Özet rapor
print("\n" + "=" * 70)
print("📋 RESNET50 ÖZET RAPORU")
print("=" * 70)
print(f"  Test Accuracy:     {test_accuracy*100:.2f}%")
print(f"  Test Loss:         {test_loss:.4f}")
print(f"  Precision:         {precision:.4f}")
print(f"  Recall:            {recall:.4f}")
print(f"  F1-Score:          {f1:.4f}")
print(f"  Model Boyutu:      {model_size:.2f} MB")
print("=" * 70)

# Sınıf bazlı detaylı rapor
class_names = list(test_generator.class_indices.keys())
print("\n📊 SINIF BAZLI PERFORMANS:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
# ============================================================
# 8. MODELLERİ DRIVE'A YEDEKLE
# ============================================================

os.makedirs(f"{drive_proje}/models", exist_ok=True)

for model_file in ['resnet50_phase1.keras', 'resnet50_final.keras']:
    src = f"models/{model_file}"
    dst = f"{drive_proje}/models/{model_file}"
    if os.path.exists(src):
        shutil.copy(src, dst)
        size_mb = os.path.getsize(dst) / (1024*1024)
        print(f"✅ {model_file} Drive'a yedeklendi ({size_mb:.1f} MB)")

print("\n📌 ResNet50 modelleri Drive'da güvende.")

## ✅ ResNet50 Eğitimi Tamamlandı

### Sonuçlar
- **Test Accuracy**: %98.65
- **F1-Score**: 0.9865
- **Model Boyutu**: 203.90 MB
- **Eğitim Süresi**: ~85 dakika (Phase 1 + Phase 2)

### Çıktılar
- `models/resnet50_final.keras` — Eğitilmiş model
- Drive yedeği: `Domates_Projesi/models/resnet50_final.keras`

### Yorum
ResNet50 derin yapısı sayesinde **en yüksek doğruluğu** elde etti. Tüm sınıflarda
tutarlı yüksek F1 skoru gösterdi. Ancak 204 MB ile mobil için ağır.

### Önemli Bulgu
ResNet50 için doğru preprocessing (`preprocess_input`) kritikti. Yanlış preprocessing
ile ilk denememizde model %20 accuracy verdi. Doğru preprocessing'le %98.65'e çıktı.
Bu, transfer learning'de **model-spesifik preprocessing** gerekliliğini gösteriyor.

### Sıradaki Adım
👉 `05_efficientnetb0_egitim.ipynb` notebook'unu açın ve modern bir mimariyi deneyin.